# Lang-Bridge end-to-end test: two videos → Hugging Face bucket → the app

Two 6 Minute English episodes (Neil and a co-host), English → Amharic, every stage, results
pushed to your bucket after each stage. Fill in `HF_TOKEN` (can write to the bucket) and
`BUCKET` (e.g. `you/lang-bridge-runs`) below, then **Run all**, and follow it in the app
(Home → Runs on …).

## Where it runs

| | Before **Run all** |
|---|---|
| **Kaggle** | Import this notebook (File → Import notebook → Link: `https://raw.githubusercontent.com/LetPastorSpeakInYourLanguage/lang_recomp/main/notebooks/lang_bridge.ipynb`); Settings → Accelerator **GPU T4 x2** (both are used), Internet **on** (phone-verified account). Use `STORAGE = bucket` to keep results. |
| **Colab** | Runtime → Change runtime type → **T4 GPU**. Drive is connected by itself when a setting points into `/content/drive/…`. |
| **Your own machine** | Jupyter with Python ≥ 3.10 (a fresh virtual environment is best). An NVIDIA GPU is used if there is one, otherwise the CPU (fine for transcripts, slow for voices). PyTorch and ffmpeg are installed if missing. |

Everything is set in the **Settings** cell, the token and bucket included.

In [ ]:
#@title Settings
BUCKET = ""  #@param {type:"string"}
#@markdown Your Hugging Face bucket, `namespace/name`.

HF_TOKEN = ""  #@param {type:"string"}
#@markdown Your [Hugging Face token](https://huggingface.co/settings/tokens) (write access if you use a bucket). **Never share or commit this notebook with a token filled in.**
DEVICE = "auto"  #@param ["auto", "cpu", "1 GPU", "2 GPUs"]
#@markdown `auto`: every GPU there is (Kaggle T4 x2 → 2, Colab T4 → 1), else the CPU.
WORK_DIR = ""  #@param {type:"string"}
#@markdown Code, models and scratch files. Empty: `/kaggle/working`, `/content`, or `~/lang-bridge` on your own machine.
CODE_BRANCH = "main"  #@param {type:"string"}

In [ ]:
#@title Get Lang-Bridge
import importlib.util, io, os, shutil, subprocess, sys, urllib.request, zipfile
from pathlib import Path

def _runtime():
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        return 'kaggle'
    try:
        if importlib.util.find_spec('google.colab'):
            return 'colab'
    except (ImportError, ValueError):
        pass
    return 'local'

RUNTIME = _runtime()
ROOT = Path(WORK_DIR).expanduser() if WORK_DIR.strip() else Path({'kaggle': '/kaggle/working', 'colab': '/content'}.get(RUNTIME) or Path.home() / 'lang-bridge')
ROOT.mkdir(parents=True, exist_ok=True)
BRANCH = CODE_BRANCH.strip() or 'main'
# On your own machine, a notebook opened from inside a Lang-Bridge checkout uses that checkout.
_here = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'worker' / 'lb_worker').is_dir() and (p / 'app').is_dir()), None)
if RUNTIME == 'local' and _here:
    CODE = _here  # as it is, on whatever branch it has checked out (CODE_BRANCH is not used)
    BRANCH = subprocess.run(['git', '-C', str(CODE), 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'checkout'
else:
    CODE = ROOT / 'lang_recomp'
    if shutil.which('git'):
        if not (CODE / '.git').exists():
            shutil.rmtree(CODE, ignore_errors=True)
            subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', BRANCH, 'https://github.com/LetPastorSpeakInYourLanguage/lang_recomp', str(CODE)], check=True)
        else:
            subprocess.run(['git', '-C', str(CODE), 'fetch', '-q', '--depth', '1', 'origin', BRANCH], check=True)
            subprocess.run(['git', '-C', str(CODE), 'checkout', '-q', '-f', '-B', BRANCH, 'FETCH_HEAD'], check=True)
    else:  # no git on this machine: the branch as a zip
        shutil.rmtree(CODE, ignore_errors=True)
        with urllib.request.urlopen('https://github.com/LetPastorSpeakInYourLanguage/lang_recomp/archive/refs/heads/' + BRANCH + '.zip') as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(ROOT)
        next(p for p in ROOT.glob('lang_recomp-*') if p.is_dir()).rename(CODE)
sys.path[:0] = [str(CODE), str(CODE / 'worker')]
_v = subprocess.run(['git', '-C', str(CODE), 'log', '-1', '--format=%h (%cd)'], capture_output=True, text=True).stdout.strip() if (CODE / '.git').exists() else 'zip'
print(f'Lang-Bridge {BRANCH} {_v} in {CODE} · running on {RUNTIME}')

In [ ]:
#@title Run the test
from lb_worker import env
from lb_worker.research import run_folder
M = env.setup(RUNTIME, ROOT, DEVICE, 'bucket', '', BUCKET, HF_TOKEN)
result = run_folder(None, M['out'], 'en', ['am'], name='E2E test', links=["https://www.youtube.com/watch?v=xwseWCSXD3Y", "https://www.youtube.com/watch?v=SC_opiKLohg"],
                    bucket=M['bucket'], hf_token=M['token'], options=M['options'])